### Messages

Messages are the basic unit of communication in LangChain.
They store the conversation between the user and the AI model.

Every message has three parts:
* Role – who sent the message. (System , User, AI)
* Content – the actual data.
* Metadata – optional extra information. (Time, Token, and other Information)

LangChain uses the same message format for all LLM providers, making it easy to switch between models.

In [ ]:
import os

from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:llama-3.3-70b-versatile")
model


In [ ]:
model.invoke("Please Tell me what is AI")

### Text Prompt

A text prompt is a simple text input given to an AI model to get a response.
It is best for one-time tasks where conversation history is not needed.


-- When should we use Text Prompts?


- You have a single question or task.
- You don't need previous conversation.
- You want simple and less complex code.


In [ ]:
response = model.invoke("What is Langchain in 40 words")
response.content

### Message Prompts

A Message Prompt is a list of messages sent to the AI model.
It is used when the model needs conversation history instead of a single text prompt.

-- Message Types


#### 1. System Message

**Definition**
A System Message gives instructions to the AI before the conversation starts.


#### 2. Human Message

**Definition**
A Human Message represents the user's input.


#### 3. AI Message

**Definition**
An AI Message is the response generated by the AI model.

In [ ]:
from langchain.messages import SystemMessage,HumanMessage,AIMessage

messages = [
    SystemMessage("You are a poetry expert"),
    HumanMessage("Who are you")
]
response = model.invoke(messages)
response.content

Other Way

In [ ]:
systemMsg = SystemMessage("You are a coding expert")
messages = [
    systemMsg,
    HumanMessage("How to write Rest API")
]
response = model.invoke(messages)
print(response.content)

In [ ]:
## Detailed info to the LLM through System message
from langchain.messages import SystemMessage, HumanMessage

system_msg = SystemMessage("""
You are a senior Node js developer with expertise in web frameworks.
Always provide code examples and explain your reasoning.
Be concise but thorough in your explanations.
""")

messages = [
    system_msg,
    HumanMessage("How do I create a REST API?")
]
response = model.invoke(messages)
print(response.content)

In [ ]:
## Message Metadata
human_msg = HumanMessage(
    content="Hello!",
    name="alice",  # Optional: identify different users
    id="msg_123",  # Optional: unique identifier for tracing
)

response = model.invoke([
  human_msg
])
print(response.content)

In [ ]:
from langchain.messages import AIMessage, SystemMessage, HumanMessage

# Create an AI message manually (e.g., for conversation history)
ai_msg = AIMessage("I'd be happy to help you with that question!")

# Add to conversation history
messages = [
    SystemMessage("You are a helpful assistant"),
    HumanMessage("Can you help me?"),
    ai_msg,  # Insert as if it came from the model
    HumanMessage("Great! What's 2+2?")
]

response = model.invoke(messages)
print(response.content)

In [ ]:
response.usage_metadata

#### 4. Tool Message

**Definition**
ToolMessage is used to send the result of a tool execution back to the AI model. The model then uses that result to generate the final response. The tool_call_id must match the tool call ID created in the AIMessage so the model knows which tool result belongs to which request.

In [ ]:
from langchain.messages import AIMessage
from langchain.messages import ToolMessage

# After a model makes a tool call
# (Here, we demonstrate manually creating the messages for brevity)
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": "call_123"
    },
     {
                      " name": "currency_converter",
            "args": {
                "amount": 100,
                "from": "USD",
                "to": "INR"
            },
            "id": "call_124"
        }],
    
)

# Execute tool and create result message
weather_result = "Sunny, 72°F"
currency = "₹8,600"

tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123"  # Must match the call ID
)
tool_message1 = ToolMessage(
    content=currency,
    tool_call_id="call_124"  # Must match the call ID
)

# Continue conversation
messages = [
    HumanMessage(" The weather of San Francisco convert 100 USD to INR?"),
    ai_message,  # Model's tool call
    tool_message,  # Tool execution result
    tool_message1
]
response = model.invoke(messages)  # Model processes the result
print(response.content)